# DATA PREPROCESSING PIPELINE
### Cleaning, Transforming, and Preparing Data for Modeling

### Impoet Libraries


In [1]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import warnings

warnings.filterwarnings('ignore')

### Load Data

In [2]:
df = pd.read_csv('../data/raw/skincare_dataset.csv')

### Step 1 : HANDLE MISSING VALUES

In [3]:
# Categorical columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)

# Numerical columns
num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)


### Step 2: Handle Outliers

In [4]:
outlier_cols = ['Age', 'MonthlyBudget_USD']

for col in outlier_cols:

    if col in df.columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Cap outliers
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

### Step 3: Feature Engineering

In [5]:
# Age Groups
if 'Age' in df.columns:

    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[0, 25, 35, 45, 55, 100],
        labels=['18-25', '26-35', '36-45', '46-55', '55+']
    )

# Budget Tiers
if 'MonthlyBudget_USD' in df.columns:

    df['BudgetTier'] = pd.cut(
        df['MonthlyBudget_USD'],
        bins=[0, 30, 75, 150, 300, 100000],
        labels=['Low', 'Medium', 'High', 'Premium', 'Luxury']
    )

# Overall Score
if (
    'ProductEffectiveness_Score' in df.columns and
    'CustomerSatisfaction_pct' in df.columns
):

    df['OverallScore'] = (
        (df['ProductEffectiveness_Score'] * 20) +
        df['CustomerSatisfaction_pct']
    ) / 2

# Active Ingredients Count
ingredient_cols = [col for col in df.columns if col.startswith('Uses')]

if len(ingredient_cols) > 0:

    df['ActiveIngredientsCount'] = df[ingredient_cols].sum(axis=1)


### Step 4: Encode Categorical Variables

In [6]:
from sklearn.preprocessing import LabelEncoder

# Create dictionary
label_encoders = {}

categorical_cols = [
    'SkinType',
    'Gender',
    'SkinConcerns',
    'AgeGroup',
    'BudgetTier'
]

for col in categorical_cols:

    if col in df.columns:

        le = LabelEncoder()

        df[col + '_encoded'] = le.fit_transform(
            df[col].astype(str)
        )

        label_encoders[col] = le


### Step 5: Scale Numerical Features

In [7]:
import os
import joblib
from sklearn.preprocessing import StandardScaler


os.makedirs('../data/models', exist_ok=True)

os.makedirs('../data/processed', exist_ok=True)

# SCALE NUMERICAL FEATURES

scale_cols = [
    'Age',
    'MonthlyBudget_USD',
    'ProductEffectiveness_Score',
    'CustomerSatisfaction_pct',
    'ActiveIngredientsCount'
]

# Keep only existing columns
scale_cols = [
    col for col in scale_cols
    if col in df.columns
]

# Initialize scaler
scaler = StandardScaler()

# Create scaled column names
scaled_col_names = [
    col + '_scaled'
    for col in scale_cols
]

# Apply scaling
df[scaled_col_names] = scaler.fit_transform(
    df[scale_cols]
)

# Save scaler
joblib.dump(
    scaler,
    '../data/models/scaler.pkl'
)

# Save processed dataset
df.to_csv(
    '../data/processed/processed_data.csv',
    index=False
)

# Display sample
display(df.head())

,UserID,Email,RegistrationDate,LastActive,Age,Gender,SkinType,SkinConcerns,FitzpatrickScale,Allergies,...,SkinType_encoded,Gender_encoded,SkinConcerns_encoded,AgeGroup_encoded,BudgetTier_encoded,Age_scaled,MonthlyBudget_USD_scaled,ProductEffectiveness_Score_scaled,CustomerSatisfaction_pct_scaled,ActiveIngredientsCount_scaled
0,DERMAI_000000,user1825@gmail.com,2026-04-18 14:54:55.322730,2026-05-05 14:54:55.763338,39.0,Female,Normal,Pigmentation,4,NaN,...,106,20,140,2,1,0.448226,-0.952357,0.887376,0.785833,-0.643655
1,DERMAI_000001,user4507@gmail.com,2024-08-21 14:54:55.322730,2026-05-13 14:54:55.763338,32.0,Male,Combination,Pigmentation,5,Essential Oils,...,35,100,140,1,1,-0.188066,-0.952357,-2.412521,-1.278152,-1.398198
2,DERMAI_000002,user3658@gmail.com,2024-12-11 14:54:55.322730,2026-04-29 14:54:55.763338,41.0,Female,Oily,Pigmentation,5,NaN,...,145,20,140,2,2,0.630024,0.341663,-0.154697,0.269837,-1.398198
3,DERMAI_000003,user1680@hotmail.com,2025-09-21 14:54:55.322730,2026-04-19 14:54:55.763338,52.0,Male,Dry,Pigmentation,1,NaN,...,91,100,140,3,2,1.629912,0.341663,-1.196770,0.166637,-0.643655
4,DERMAI_000004,user8936@gmail.com,2024-07-13 14:54:55.322730,2026-04-26 14:54:55.763338,31.0,Female,Oily,Dryness,4,NaN,...,145,20,68,1,1,-0.278965,-0.952357,-2.065164,-0.658957,-0.643655


### Step 6: Remove Duplicate

In [8]:
df = df.drop_duplicates()


### Step 7 : Save Processed Data

In [9]:
# Save processed dataset
processed_path = '../data/processed/processed_data.csv'

df.to_csv(processed_path, index=False)

# Train-Test Split
if 'WillRepurchase' in df.columns:

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df['WillRepurchase']
    )

    train_df.to_csv('../data/processed/train_data.csv', index=False)
    test_df.to_csv('../data/processed/test_data.csv', index=False)

# Save encoders
joblib.dump(label_encoders, '../data/models/label_encoders.pkl')


['../data/models/label_encoders.pkl']

### DISPLAY FINAL DATA

In [10]:
display(df.head())

,UserID,Email,RegistrationDate,LastActive,Age,Gender,SkinType,SkinConcerns,FitzpatrickScale,Allergies,...,SkinType_encoded,Gender_encoded,SkinConcerns_encoded,AgeGroup_encoded,BudgetTier_encoded,Age_scaled,MonthlyBudget_USD_scaled,ProductEffectiveness_Score_scaled,CustomerSatisfaction_pct_scaled,ActiveIngredientsCount_scaled
0,DERMAI_000000,user1825@gmail.com,2026-04-18 14:54:55.322730,2026-05-05 14:54:55.763338,39.0,Female,Normal,Pigmentation,4,NaN,...,106,20,140,2,1,0.448226,-0.952357,0.887376,0.785833,-0.643655
1,DERMAI_000001,user4507@gmail.com,2024-08-21 14:54:55.322730,2026-05-13 14:54:55.763338,32.0,Male,Combination,Pigmentation,5,Essential Oils,...,35,100,140,1,1,-0.188066,-0.952357,-2.412521,-1.278152,-1.398198
2,DERMAI_000002,user3658@gmail.com,2024-12-11 14:54:55.322730,2026-04-29 14:54:55.763338,41.0,Female,Oily,Pigmentation,5,NaN,...,145,20,140,2,2,0.630024,0.341663,-0.154697,0.269837,-1.398198
3,DERMAI_000003,user1680@hotmail.com,2025-09-21 14:54:55.322730,2026-04-19 14:54:55.763338,52.0,Male,Dry,Pigmentation,1,NaN,...,91,100,140,3,2,1.629912,0.341663,-1.196770,0.166637,-0.643655
4,DERMAI_000004,user8936@gmail.com,2024-07-13 14:54:55.322730,2026-04-26 14:54:55.763338,31.0,Female,Oily,Dryness,4,NaN,...,145,20,68,1,1,-0.278965,-0.952357,-2.065164,-0.658957,-0.643655
